<a href="https://colab.research.google.com/github/Stdcoders/Graph-RAG/blob/main/GraphRAG_L4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!git clone https://github.com/genaiconference/Agentic_KAG_Workshop_DHS_2026.git

Cloning into 'Agentic_KAG_Workshop_DHS_2026'...
remote: Enumerating objects: 403, done.
remote: Counting objects: 100% (179/179), done.
remote: Compressing objects: 100% (123/123), done.
remote: Total 403 (delta 125), reused 57 (delta 55), pack-reused 224 (from 2)
Receiving objects: 100% (403/403), 13.34 MiB | 16.13 MiB/s, done.
Resolving deltas: 100% (217/217), done.


In [ ]:
!pip install langchain-neo4j langchain-nvidia-ai-endpoints langgraph --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 2.4 MB/s eta 0:00:00


In [ ]:
import os

os.chdir('/content/Agentic_KAG_Workshop_DHS_2026/')

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    print("error reading env details")
    pass

# --- Neo4j Sandbox ---
NEO4J_URI      = os.getenv('NEO4J_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE') or 'neo4j'

# --- OpenAI ---
os.environ.setdefault(
    'NVIDIA_API_KEY',
    os.getenv('NVIDIA_API_KEY')
)

print('NEO4J_URI :', NEO4J_URI)

NEO4J_URI : neo4j+s://313964e6.databases.neo4j.io


In [ ]:

from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
driver.verify_connectivity()
print('Connected to Neo4j ✔')

Connected to Neo4j ✔


In [ ]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA

chat_llm = ChatNVIDIA(model="meta/llama-3.1-8b-instruct", temperature=0)

In [ ]:
from langchain_neo4j import Neo4jGraph
from IPython.display import Markdown

graph = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE,
    enhanced_schema=True,   # richer schema (property types, example values) improves generation
)

# Refresh + inspect the schema the LLM will use as context
graph.refresh_schema()
Markdown(graph.schema)

Node properties:
- **Skill**
  - `name`: STRING Example: "Python"
- **Cluster**
  - `name`: STRING Example: "Programming Languages"
- **Document**
  - `path`: STRING Available options: ['/content/Leave_Policy.pdf']
  - `createdAt`: STRING Available options: ['2026-08-10T09:27:20.290103+00:00']
  - `document_type`: STRING Available options: ['pdf']
- **Chunk**
  - `index`: INTEGER Min: 0, Max: 2
  - `text`: STRING Available options: ['Page 1 of 7  Back to Contents  LEA VE POLICY  Obje', ' require the approval of the division  management ', '  avail 2 days leave within a period of 3 months o']
- **Policy**
  - `name`: STRING Available options: ['Parental Leave Policy - India', 'Leave Without Pay (LWP) Policy', 'Transition/Relocation Leave Policy', 'Marriage Leave Policy', 'Bereavement Leave Policy', 'Compensatory Offs Policy']
  - `title`: STRING Available options: ['Compensatory Offs', 'Periodical Review of Leave Records']
- **Entity**
  - `name`: STRING Available options: ['Associate', 'Birthing mother', 'Non-birthing parent', 'Intern', 'Immediate Manager', 'SLT member', 'HR', 'Division Management Team']
- **Concept**
  - `name`: STRING Available options: ['Child’s birth/surrogacy/adoption', 'Medical examination', 'Sick leave', 'Loss of Pay (LOP)', 'Leave Without Pay (LWP)', 'Unauthorized absence', 'Relocation Leave', 'Marriage Leave', 'Bereavement Leave', 'Compensatory Offs']
- **Objective**
  - `description`: STRING Available options: ['To meet business exigency by providing compensator']
- **Eligibility**
  - `criteria`: STRING Available options: ['All associates at GJFA 5 to 8']
- **Procedure**
  - `step`: STRING Available options: ['Associate should have worked at least 8 hours on a']
- **Condition**
  - `rule`: STRING Available options: ['Compensatory off must be availed within two months', 'Compensatory off cannot be taken in advance', 'Compensatory off will not be carried forward, accu', 'Maximum 2 compensatory offs can be availed in a mo']
- **Responsibility**
  - `description`: STRING Available options: ['Ensure leave records of their respective teams are']
  - `role`: STRING Available options: ['Manager']
- **Action**
  - `description`: STRING Available options: ['Bring discrepancies to the notice of the HRBP', 'HRBP initiates corrections by sending to HR Servic', 'Ensure unrecorded leaves are recorded on HR Core']
- **Requirement**
  - `description`: STRING Available options: ['Maintain sufficient evidences of periodical review']
Relationship properties:
- **ELIGIBLE_FOR**
  - `condition`: STRING Available options: ['Following birth, surrogacy or adoption of a child', 'Only in exceptional circumstances, at discretion o', 'Relocating from other cities at time of joining/in', 'At time of own marriage', 'Death of parent, sibling, spouse or child', 'To meet business exigency', 'Subject to Manager’s approval']
  - `during`: STRING Available options: ['Internship period']
  - `frequency`: STRING Available options: ['Maximum one day per month']
- **APPLIES_WITHIN**
  - `duration`: STRING Available options: ['one year']
- **APPROVES**
  - `condition`: STRING 
- **SUBJECT_TO**
  - `condition`: STRING 
- **ENTITLES**
  - `duration`: STRING 
  - `timeframe`: STRING 
  - `condition`: STRING 
- **MAY_REQUEST**
  - `condition`: STRING 
  - `by`: STRING 
- **CAN_LEAD_TO**
  - `condition`: STRING 
- **SUBJECT_TO_RULES_OF**
  - `policy`: STRING 
- **MUST_OFFER_EXPLANATION_TO**
  - `context`: STRING 
The relationships:
(:Skill)-[:IN_CLUSTER]->(:Cluster)
(:Skill)-[:IS_TYPE_OF]->(:Skill)
(:Skill)-[:ADJACENT_TO]->(:Skill)
(:Skill)-[:IS_PREREQUISITE_FOR]->(:Skill)
(:Skill)-[:IS_EQUIVALENT_TO]->(:Skill)
(:Chunk)-[:FROM_DOCUMENT]->(:Document)
(:Chunk)-[:NEXT_CHUNK]->(:Chunk)
(:Policy)-[:APPLIES_WITHIN]->(:Concept)
(:Policy)-[:FROM_CHUNK]->(:Chunk)
(:Policy)-[:HAS_ELIGIBILITY]->(:Eligibility)
(:Policy)-[:HAS_PROCEDURE]->(:Procedure)
(:Policy)-[:HAS_CONDITION]->(:Condition)
(:Policy)-[:HAS_OBJECTIVE]->(:Objective)
(:Policy)-[:HAS_RESPONSIBILITY]->(:Responsibility)
(:Entity)-[:ELIGIBLE_FOR]->(:Policy)
(:Entity)-[:ELIGIBLE_FOR]->(:Concept)
(:Entity)-[:SUBJECT_TO_RULES_OF]->(:Concept)
(:Entity)-[:MUST_OFFER_EXPLANATION_TO]->(:Entity)
(:Entity)-[:GOVERNED_BY]->(:Policy)
(:Entity)-[:FROM_CHUNK]->(:Chunk)
(:Entity)-[:REPORTS_TO]->(:Entity)
(:Entity)-[:SUBJECT_TO]->(:Concept)
(:Entity)-[:APPROVES]->(:Concept)
(:Entity)-[:REQUESTS_EXPLANATION_FROM]->(:Entity)
(:Entity)-[:REQUIRES_APPROVAL_OF]->(:Concept)
(:Entity)-[:MAY_REQUEST]->(:Concept)
(:Concept)-[:FROM_CHUNK]->(:Chunk)
(:Concept)-[:CAN_LEAD_TO]->(:Concept)
(:Concept)-[:REQUIRES_PRIOR_EXHAUSTION_OF]->(:Concept)
(:Concept)-[:ENTITLES]->(:Entity)
(:Objective)-[:FROM_CHUNK]->(:Chunk)
(:Eligibility)-[:FROM_CHUNK]->(:Chunk)
(:Procedure)-[:FROM_CHUNK]->(:Chunk)
(:Condition)-[:FROM_CHUNK]->(:Chunk)
(:Responsibility)-[:FROM_CHUNK]->(:Chunk)
(:Responsibility)-[:REQUIRES_ACTION]->(:Action)
(:Responsibility)-[:REQUIRES_ACTION]->(:Requirement)
(:Action)-[:FROM_CHUNK]->(:Chunk)
(:Action)-[:LEADS_TO]->(:Action)
(:Requirement)-[:FROM_CHUNK]->(:Chunk)

In [ ]:
from langchain_neo4j import GraphCypherQAChain
from langchain_core.prompts.prompt import PromptTemplate

CYPHER_GENERATION_TEMPLATE = """Task:
Generate Cypher statement to query a graph database.

Instructions:
Use only the provided relationship types and properties in the schema.
Do not use any other relationship types or properties that are not provided.

Schema:
{schema}

Note: Do not include any explanations or apologies in your responses.
Do not respond to any questions that might ask anything else than for you to construct a Cypher statement.
Do not include any text except the generated Cypher statement.

Examples: Here are a few examples of generated Cypher statements for particular questions:
# How many people played in Top Gun?
MATCH (m:Movie {{name:"Top Gun"}})<-[:ACTED_IN]-()
RETURN count(*) AS numberOfActors

The question is:
{question}"""

CYPHER_GENERATION_PROMPT = PromptTemplate(
    input_variables=["schema", "question"], template=CYPHER_GENERATION_TEMPLATE
)

chain = GraphCypherQAChain.from_llm(
    llm=chat_llm,
    graph=graph,
    verbose=True,                 # print the generated Cypher + intermediate steps
    cypher_prompt=CYPHER_GENERATION_PROMPT,
    return_intermediate_steps=True,
    allow_dangerous_requests=True,
)
print('GraphCypherQAChain ready ✔')

GraphCypherQAChain ready ✔


In [ ]:
result = chain.invoke({"query": "List the top 10 highest-rated Hindi movies"})

print("\n=== Generated Cypher ===")
for step in result.get("intermediate_steps", []):
    if "query" in step:
        print(step["query"])

print("\n=== Final Answer ===")
print(result["result"])



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (m:Movie)<-[:RATED]-(:Critic)-[:RATED]->(m2:Movie {language:"Hindi"}) 
WHERE m2.rating > m.rating 
RETURN m2 ORDER BY m2.rating DESC LIMIT 10


Full Context:
[]

> Finished chain.

=== Generated Cypher ===
MATCH (m:Movie)<-[:RATED]-(:Critic)-[:RATED]->(m2:Movie {language:"Hindi"}) 
WHERE m2.rating > m.rating 
RETURN m2 ORDER BY m2.rating DESC LIMIT 10

=== Final Answer ===
I don't know the answer.
